# MIRI MRS: 3D spline model of a star and BreadsPSF

This tutorial builds an empirical PSF model of MIRI MRS (a *BreadsPSF*) from the observation of a bright star with many
dithers. The BreadsPSF can then replace STPSF (WebbPSF) in the PSF fits, cube extraction and forward modeling of breads.

The steps are:

1. continuum-normalized spectrum of the star, combined over all the exposures;
2. centroid of the star in the sequence, by fitting STPSF at each wavelength;
3. 3D spline model of the star: a B-spline in (x, y, λ), with x and y the IFU coordinates relative to the star scaled
   by λ_ref/λ, fitted to the point cloud of all the detector pixels of all the exposures;
4. BreadsPSF: the 3D spline evaluated on a regular grid at each wavelength of the band.

We use the calibration star **10 Lac** (program 3779) in channel **1A** (MIRIFUSHORT, `SHORT` band), observed with 18
groups of 4 dithers. The star is in the field of view, and the small offsets between the groups sample the PSF below
the size of a pixel. To keep the run time short, we only use 4 groups (16 exposures) here.

The inputs are the stage 2 `*_mirifushort_cal.fits` files, e.g. from `run_stage2_miri` (see the MIRI MRS calibration
tutorial).

## Environment

The thread limits must be set **before** numpy is imported: the parallelization is done with processes.

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ.setdefault("CRDS_PATH", os.path.expanduser("~/crds_cache"))
os.environ.setdefault("CRDS_SERVER_URL", "https://jwst-crds.stsci.edu")
# os.environ["BREADS_DATA"] = "/path/to/breads_data"   # the BreadsPSF is written to $BREADS_DATA/BreadsPSF/
assert os.getenv("BREADS_DATA") is not None, "Define BREADS_DATA"

from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import multiprocessing as mp
from astropy.io import fits

from breads.jwst_reduction.reduction_utils import compute_normalized_stellar_spectrum_miri
from breads.jwst_reduction.reduction_utils import recenter_coordinates_of_sequence_miri
from breads.jwst_reduction.reduction_utils import compute_3dsplines_miri
from breads.jwst_tools.breadspsf import create_BreadsPSF
from breads.jwst_tools.default_miri import wv_sampling_dict, wv_ref_dict

## Inputs

`CHANNEL` is the MRS channel ('1' to '4') and `CHANNEL_BAND` the channel and sub-band ('1A'). Channels 1 and 2 are on
the same MIRIFUSHORT detector: the same cal files are used for 1A and 2A, and `channel` selects the half of the
detector.

In [ ]:
DATA_DIR = "/stow/abidot/miri_data/10Lac/12A/stage2"            # stage 2 cal files
UTILS_DIR = "/stow/abidot/miri_data/10Lac/utils_tutorial_breadspsf/1A"   # intermediate products (new directory)
SIMBAD_NAME = "10 Lac"
CHANNEL, CHANNEL_BAND = "1", "1A"
BREADSPSF_NAME = "10Lac_3779_1A_tutorial.fits"                  # written to $BREADS_DATA/BreadsPSF/
GROUPS = ["03102", "0310e", "0310k", "0310u"]                   # 4 groups of 4 dithers (all 18 for the full PSF)
NUMTHREADS = 16

cal_files = sorted(f for f in glob(os.path.join(DATA_DIR, "jw*_mirifushort_cal.fits"))
                   if os.path.basename(f).split("_")[1] in GROUPS)
os.makedirs(UTILS_DIR, exist_ok=True)
wv_sampling = wv_sampling_dict[CHANNEL_BAND]
wv_ref = wv_ref_dict[CHANNEL_BAND]
prefix = os.path.basename(cal_files[0]).split("_")[0]
print(f"{len(cal_files)} exposures, band {CHANNEL_BAND}: {wv_sampling[0]:.3f}-{wv_sampling[-1]:.3f} um, "
      f"{wv_sampling.size} channels, wv_ref = {wv_ref} um")

mypool = mp.Pool(processes=NUMTHREADS)

## 1. Continuum-normalized spectrum of the star

The spectrum of the star, normalized by its continuum, is combined over all the exposures. It is used in the next
steps to model the stellar lines, so that the splines only have to fit the spatial and slow spectral variations of the
PSF.

In [ ]:
contnorm_filename = os.path.join(UTILS_DIR, prefix + "_starspec_contnorm_combined_1dspline.fits")
compute_normalized_stellar_spectrum_miri(cal_files, CHANNEL, UTILS_DIR, contnorm_filename, mppool=mypool,
                                         targetname=SIMBAD_NAME, overwrite=False)

with fits.open(contnorm_filename) as hdul:
    plt.figure(figsize=(10, 3))
    plt.plot(hdul["WAVE"].data, hdul["COM_FLUXES"].data, lw=0.8)
plt.xlabel(r"$\lambda$ ($\mu$m)")
plt.ylabel("normalized flux")
plt.title(f"{SIMBAD_NAME}: continuum-normalized spectrum, {CHANNEL_BAND}")
plt.show()

## 2. Centroid of the star

The star is located at each wavelength by fitting STPSF to the pixels within `OWA` of the star (here 1 arcsec), on all
the exposures of the sequence together. The centroids are then fitted with a polynomial of the wavelength, which defines
the position of the star used in the next steps.

Computing STPSF for the band takes a while the first time; it is saved in `UTILS_DIR` and reloaded afterwards.

In [ ]:
IWA, OWA = 0.0, 1.0
fitpsf_filename = os.path.join(UTILS_DIR, prefix + "_fitspsf_stpsf.fits")
poly_centroid_filename, _ = recenter_coordinates_of_sequence_miri(
    cal_files, CHANNEL, UTILS_DIR, contnorm_filename, fitpsf_filename,
    init_centroid=[0.0, 0.0], IWA=IWA, OWA=OWA, mppool=mypool, overwrite=False,
    targetname=SIMBAD_NAME, use_stpsf=True)

p_ra, p_dec = np.loadtxt(poly_centroid_filename)
centroid = (np.polyval(p_ra, wv_ref), np.polyval(p_dec, wv_ref))
print(f"Centroid of the star at {wv_ref} um: ({centroid[0]:+.4f}, {centroid[1]:+.4f}) arcsec")

`recenter_coordinates_of_sequence_miri` also saves a diagnostic plot of the centroids and fluxes fitted at each
wavelength:

In [ ]:
from IPython.display import Image
Image(fitpsf_filename.replace(".fits", "_results.png"))

## 3. 3D spline model of the star

The star is modeled by a 3D B-spline in (x, y, λ). The spatial coordinates are the IFU coordinates relative to the
centroid of step 2, scaled by λ_ref/λ, so that the diffraction pattern is (to first order) independent of λ; the
stellar lines are taken out with the normalized spectrum of step 1. The spline is fitted in overlapping stamps of
`stamp_size` arcsec.

The spacing of the spatial nodes sets the resolution of the model: it must be small compared to the FWHM (about 0.3
arcsec in 1A), but each node must be constrained by enough pixels. We use 0.1 arcsec here, over ±2.5 arcsec;
the full set of 72 exposures allows smaller spacings (0.05 arcsec). The default wavelength nodes are 5 nodes over the
band (`default_miri.wv_nodes_3D_dict`).

`coords_filename_filter` with `centroid_per_frame=False` applies the centroid of the sequence (step 2) to all the
exposures.

In [ ]:
x_nodes = np.arange(-2.5, 2.5001, 0.1)
y_nodes = np.arange(-2.5, 2.5001, 0.1)
coords_filter = os.path.basename(poly_centroid_filename).replace(prefix, "")

combdataobj = compute_3dsplines_miri(cal_files, CHANNEL, UTILS_DIR, SIMBAD_NAME, contnorm_filename,
                                     x_nodes=x_nodes, y_nodes=y_nodes,
                                     coords_filename_filter=coords_filter, centroid_per_frame=False,
                                     spline3d_suffix="_tutorial", numthreads=NUMTHREADS,
                                     save_pickle=True, load_pickle=True)
spline3d_filename = combdataobj.default_filenames["compute_starsubtraction_3dspline"]
print(spline3d_filename)

The point cloud fitted by the spline: all the valid pixels of all the exposures in a thin slice of wavelength, in the
scaled IFU coordinates. The small offsets between the groups fill the gaps between the pixels of a single exposure.

In [ ]:
ifux, ifuy = combdataobj.get_ifu_coords()
wv = combdataobj.wavelengths
lam0 = 5.3
sl = (np.abs(wv - lam0) < 0.001) & np.isfinite(combdataobj.bad_pixels) & np.isfinite(combdataobj.data)
sx, sy = ifux[sl] * wv_ref / wv[sl], ifuy[sl] * wv_ref / wv[sl]

fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(sx, sy, c=np.log10(np.clip(combdataobj.data[sl], 1e-3 * np.nanmax(combdataobj.data[sl]), None)),
                s=4, cmap="magma")
fig.colorbar(sc, ax=ax, label="log10 flux (MJy/sr)")
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_aspect("equal")
ax.set_xlabel(r"IFU x $\times\ \lambda_{ref}/\lambda$ (arcsec)")
ax.set_ylabel(r"IFU y $\times\ \lambda_{ref}/\lambda$ (arcsec)")
ax.set_title(f"{sl.sum()} pixels at {lam0} ± 0.001 um")
plt.show()

## 4. BreadsPSF

The BreadsPSF is the 3D spline evaluated on a regular grid at each wavelength of `wv_sampling` and saved in
`$BREADS_DATA/BreadsPSF/`. The grid must be square (`x_vec` equal to `y_vec`) and should not extend beyond the region
constrained by the data: nodes without data have no constraint and give unreliable values there.

Without a reference spectrum of the star (`stis_spectrum=None`), the BreadsPSF is not flux calibrated: it gives the
shape of the PSF in MJy/sr, normalized by the spectrum of 10 Lac. With a CALSPEC spectrum, it would be in
[MJy/sr]/[1 MJy], and the amplitudes fitted with it would be fluxes in MJy.

In [ ]:
vec_psf = np.arange(-1.5, 1.5001, 0.02)
create_BreadsPSF(spline3d_filename, vec_psf, vec_psf, wv_sampling, basename=BREADSPSF_NAME,
                 numthreads=NUMTHREADS, stis_spectrum=None, overwrite=True)

with fits.open(os.path.join(os.environ["BREADS_DATA"], "BreadsPSF", BREADSPSF_NAME)) as hdul:
    print([h.name for h in hdul])
    epsfs, psf_x, psf_y, psf_wv = hdul["EPSFS"].data, hdul["X"].data, hdul["Y"].data, hdul["WAVE"].data
print(epsfs.shape)

Slices of the BreadsPSF at three wavelengths, and its azimuthal profile. The first dark ring moves outwards with λ:
the grid is in arcsec on the sky, not scaled by λ.

In [ ]:
ids = [50, epsfs.shape[0] // 2, epsfs.shape[0] - 50]
extent = [psf_x[0, 0], psf_x[0, -1], psf_y[0, 0], psf_y[-1, 0]]
fig, axes = plt.subplots(1, 4, figsize=(20, 4.8))
for ax, i in zip(axes[:3], ids):
    im = epsfs[i] / np.nanmax(epsfs[i])
    h = ax.imshow(np.log10(np.clip(im, 1e-4, None)), origin="lower", extent=extent, cmap="magma", vmin=-4, vmax=0)
    ax.set_title(f"{psf_wv[i]:.3f} um")
    ax.set_xlabel("IFU x (arcsec)")
fig.colorbar(h, ax=axes[:3], label="log10 (normalized PSF)", shrink=0.8)
axes[0].set_ylabel("IFU y (arcsec)")
r = np.hypot(psf_x, psf_y)
bins = np.arange(0, 1.5, 0.02)
for i in ids:
    im = epsfs[i] / np.nanmax(epsfs[i])
    prof = [np.nanmean(im[(r >= a) & (r < a + 0.02)]) for a in bins]
    axes[3].semilogy(bins + 0.01, prof, label=f"{psf_wv[i]:.3f} um")
axes[3].set_xlabel("radius (arcsec)")
axes[3].set_ylabel("azimuthal mean (normalized)")
axes[3].legend()
plt.show()

The BreadsPSF can now be used instead of STPSF, by passing its file name to `use_breadspsf` (with `use_stpsf=False`)
in `recenter_coordinates_per_frame_miri`, `cube_extraction_miri` or `fitpsf`. A common refinement is to recenter each
exposure with this first BreadsPSF, then to fit the 3D spline again with the centroids of each exposure
(`centroid_per_frame=True`) to get a sharper PSF.

In [ ]:
mypool.close()
mypool.join()